# IUCN Status Comparison Analysis

This notebook compares `iucn_status.json` and `iucn_status_new.json` to identify species with updated IUCN conservation statuses.

## Import Required Libraries

In [1]:
import json
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional

## Load Data Files

In [2]:
# Define file paths
old_file = Path('../../lib/iucn_status.json')
new_file = Path('../../lib/iucn_status_new.json')

# Load JSON files
with open(old_file, 'r', encoding='utf-8') as f:
    old_data = json.load(f)

with open(new_file, 'r', encoding='utf-8') as f:
    new_data = json.load(f)

print(f"Old data contains {len(old_data)} species")
print(f"New data contains {len(new_data)} species")

Old data contains 1026 species
New data contains 1025 species


## Helper Functions

In [3]:
def get_iucn_status(species: Dict) -> Optional[str]:
    """Extract IUCN status from species data."""
    if 'laws' in species:
        for law in species['laws']:
            if law.get('name', {}).get('en') == 'IUCN':
                return law.get('value')
    return None

def get_iucn_url(species: Dict) -> Optional[str]:
    """Extract IUCN URL from species data."""
    if 'laws' in species:
        for law in species['laws']:
            if law.get('name', {}).get('en') == 'IUCN':
                return law.get('note', '')
    return None

def get_scientific_name(species: Dict) -> str:
    """Extract scientific name from species data."""
    return species.get('scientific_name', {}).get('value', '')

def get_common_name_en(species: Dict) -> str:
    """Extract English common name from species data."""
    return species.get('common_name_en', {}).get('value', '')

def get_taxonomic_info(species: Dict) -> Dict[str, str]:
    """Extract taxonomic information from species data."""
    return {
        'kingdom': species.get('kingdom_latin', ''),
        'phylum': species.get('phylum_latin', ''),
        'class': species.get('class_latin', ''),
        'order': species.get('order_latin', ''),
        'family': species.get('family_latin', '')
    }

## Create Species Dictionaries for Quick Lookup

In [4]:
# Create dictionaries with scientific name as key
old_species_dict = {get_scientific_name(sp): sp for sp in old_data}
new_species_dict = {get_scientific_name(sp): sp for sp in new_data}

print(f"Old species dictionary: {len(old_species_dict)} unique species")
print(f"New species dictionary: {len(new_species_dict)} unique species")

Old species dictionary: 1026 unique species
New species dictionary: 1025 unique species


## Compare IUCN Statuses

In [5]:
# Find species with updated statuses
updated_species = []
new_species = []
removed_species = []
unchanged_species = []

# Check all species in the new file
for sci_name, new_sp in new_species_dict.items():
    new_status = get_iucn_status(new_sp)
    
    if sci_name in old_species_dict:
        old_sp = old_species_dict[sci_name]
        old_status = get_iucn_status(old_sp)
        
        if old_status != new_status:
            taxonomic_info = get_taxonomic_info(new_sp)
            updated_species.append({
                'scientific_name': sci_name,
                'common_name_en': get_common_name_en(new_sp),
                'old_status': old_status,
                'new_status': new_status,
                'kingdom': taxonomic_info['kingdom'],
                'phylum': taxonomic_info['phylum'],
                'class': taxonomic_info['class'],
                'order': taxonomic_info['order'],
                'family': taxonomic_info['family'],
                'iucn_url': get_iucn_url(new_sp)
            })
        else:
            unchanged_species.append(sci_name)
    else:
        # Species is new in the updated file
        taxonomic_info = get_taxonomic_info(new_sp)
        new_species.append({
            'scientific_name': sci_name,
            'common_name_en': get_common_name_en(new_sp),
            'status': new_status,
            'kingdom': taxonomic_info['kingdom'],
            'phylum': taxonomic_info['phylum'],
            'class': taxonomic_info['class'],
            'order': taxonomic_info['order'],
            'family': taxonomic_info['family'],
            'iucn_url': get_iucn_url(new_sp)
        })

# Check for removed species (in old but not in new)
for sci_name, old_sp in old_species_dict.items():
    if sci_name not in new_species_dict:
        taxonomic_info = get_taxonomic_info(old_sp)
        removed_species.append({
            'scientific_name': sci_name,
            'common_name_en': get_common_name_en(old_sp),
            'old_status': get_iucn_status(old_sp),
            'kingdom': taxonomic_info['kingdom'],
            'phylum': taxonomic_info['phylum'],
            'class': taxonomic_info['class'],
            'order': taxonomic_info['order'],
            'family': taxonomic_info['family']
        })

print(f"\nSummary:")
print(f"  Species with updated status: {len(updated_species)}")
print(f"  New species added: {len(new_species)}")
print(f"  Species removed: {len(removed_species)}")
print(f"  Species unchanged: {len(unchanged_species)}")


Summary:
  Species with updated status: 15
  New species added: 9
  Species removed: 10
  Species unchanged: 1001


## Display Updated Species

In [6]:
# Create DataFrame for updated species
if updated_species:
    df_updated = pd.DataFrame(updated_species)
    print(f"\n=== Species with Updated IUCN Status ({len(df_updated)}) ===")
    display(df_updated)
else:
    print("No species with updated status found.")


=== Species with Updated IUCN Status (15) ===


,scientific_name,common_name_en,old_status,new_status,kingdom,phylum,class,order,family,iucn_url
0,Altigena tonkinensis,Cá Hoả,VU,NT,ANIMALIA,CHORDATA,ACTINOPTERYGII,CYPRINIFORMES,CYPRINIDAE,https://www.iucnredlist.org/species/167009/214...
1,Belomys pearsonii,Hairy-footed Flying Squirrel,DD,LC,ANIMALIA,CHORDATA,MAMMALIA,RODENTIA,SCIURIDAE,https://www.iucnredlist.org/species/2756/26903...
2,Castanopsis kawakamii,Ca Oi Qua To,LR/nt,NT,PLANTAE,TRACHEOPHYTA,MAGNOLIOPSIDA,FAGALES,FAGACEAE,https://www.iucnredlist.org/species/32387/9695771
3,Chelonia mydas,Green Turtle,EN,LC,ANIMALIA,CHORDATA,REPTILIA,TESTUDINES,CHELONIIDAE,https://www.iucnredlist.org/species/4615/28510...
4,Diospyros mun,Ebony,CR,EN,PLANTAE,TRACHEOPHYTA,MAGNOLIOPSIDA,ERICALES,EBENACEAE,https://www.iucnredlist.org/species/32821/2824811
5,Disepalum petelotii,Nhoc trai khop la mac,LR/lc,LC,PLANTAE,TRACHEOPHYTA,MAGNOLIOPSIDA,MAGNOLIALES,ANNONACEAE,https://www.iucnredlist.org/species/32823/9732714
6,Drepananthus filiformis,,LR/lc,LC,PLANTAE,TRACHEOPHYTA,MAGNOLIOPSIDA,MAGNOLIALES,ANNONACEAE,https://www.iucnredlist.org/species/35949/9969792
7,Ephippiorhynchus asiaticus,Black-necked Stork,NT,LC,ANIMALIA,CHORDATA,AVES,CICONIIFORMES,CICONIIDAE,https://www.iucnredlist.org/species/22697702/2...
8,Ketupa zeylonensis,Brown Fish-owl,EN,LC,ANIMALIA,CHORDATA,AVES,STRIGIFORMES,STRIGIDAE,https://www.iucnredlist.org/species/22689012/2...
9,Limosa lapponica,Bar-tailed Godwit,LC,NT,ANIMALIA,CHORDATA,AVES,CHARADRIIFORMES,SCOLOPACIDAE,https://www.iucnredlist.org/species/22693158/2...


## Display New Species

In [7]:
# Create DataFrame for new species
if new_species:
    df_new = pd.DataFrame(new_species)
    print(f"\n=== New Species Added ({len(df_new)}) ===")
    display(df_new)
else:
    print("No new species found.")


=== New Species Added (9) ===


,scientific_name,common_name_en,status,kingdom,phylum,class,order,family,iucn_url
0,Calamus acanthospathus,Wai Hom,LC,PLANTAE,TRACHEOPHYTA,LILIOPSIDA,ARECALES,ARECACEAE,https://www.iucnredlist.org/species/111454424/...
1,Calamus inermis,Takat,LC,PLANTAE,TRACHEOPHYTA,LILIOPSIDA,ARECALES,ARECACEAE,https://www.iucnredlist.org/species/253573694/...
2,Calamus lateralis,May Tu,EN,PLANTAE,TRACHEOPHYTA,LILIOPSIDA,ARECALES,ARECACEAE,https://www.iucnredlist.org/species/191588/199...
3,Callosciurus honkhoaiensis,Hon Khoai Squirrel,CR,ANIMALIA,CHORDATA,MAMMALIA,RODENTIA,SCIURIDAE,https://www.iucnredlist.org/species/277177325/...
4,Cibotium barometz,Lamb of Tartary,LC,PLANTAE,TRACHEOPHYTA,POLYPODIOPSIDA,CYATHEALES,CIBOTIACEAE,https://www.iucnredlist.org/species/88305249/8...
5,Diospyros mollis,Ma Kleua,LC,PLANTAE,TRACHEOPHYTA,MAGNOLIOPSIDA,ERICALES,EBENACEAE,https://www.iucnredlist.org/species/173958/140...
6,Falco severus,Oriental Hobby,LC,ANIMALIA,CHORDATA,AVES,FALCONIFORMES,FALCONIDAE,https://www.iucnredlist.org/species/22696470/1...
7,Rauvolfia serpentina,Indian Snakeroot,NT,PLANTAE,TRACHEOPHYTA,MAGNOLIOPSIDA,GENTIANALES,APOCYNACEAE,https://www.iucnredlist.org/species/191068/196...
8,Stereospermum binhchauense,,VU,PLANTAE,TRACHEOPHYTA,MAGNOLIOPSIDA,LAMIALES,BIGNONIACEAE,https://www.iucnredlist.org/species/213301331/...


## Display Removed Species

In [8]:
# Create DataFrame for removed species
if removed_species:
    df_removed = pd.DataFrame(removed_species)
    print(f"\n=== Species Removed ({len(df_removed)}) ===")
    display(df_removed)
else:
    print("No removed species found.")


=== Species Removed (10) ===


,scientific_name,common_name_en,old_status,kingdom,phylum,class,order,family
0,Nebrius ferrugineus,Tawny Nurse Shark,VU,ANIMALIA,CHORDATA,CHONDRICHTHYES,ORECTOLOBIFORMES,GINGLYMOSTOMATIDAE
1,Nomascus siki,Southern White-cheeked Gibbon,CR,ANIMALIA,CHORDATA,MAMMALIA,PRIMATES,HYLOBATIDAE
2,Nyctereutes procyonoides,Raccoon Dog,NA,ANIMALIA,CHORDATA,MAMMALIA,CARNIVORA,CANIDAE
3,Paphiopedilum villosum,Villose Paphiopedilum,VU,PLANTAE,TRACHEOPHYTA,LILIOPSIDA,ASPARAGALES,ORCHIDACEAE
4,Phalacrocorax carbo,Great Cormorant,LC,ANIMALIA,CHORDATA,AVES,SULIFORMES,PHALACROCORACIDAE
5,Physogyra lichtensteini,,LC,ANIMALIA,CNIDARIA,ANTHOZOA,SCLERACTINIA,PLEROGYRIDAE
6,Pteropus hypomelanus,Island Flying Fox,NT,ANIMALIA,CHORDATA,MAMMALIA,CHIROPTERA,PTEROPODIDAE
7,Pterorhinus sannio,White-browed Laughingthrush,LC,ANIMALIA,CHORDATA,AVES,PASSERIFORMES,LEIOTRICHIDAE
8,Ptyas korros,Javan Rat Snake,NT,ANIMALIA,CHORDATA,REPTILIA,SQUAMATA,COLUBRIDAE
9,Schistura spiloptera,,CR,ANIMALIA,CHORDATA,ACTINOPTERYGII,CYPRINIFORMES,NEMACHEILIDAE


## Analyze Status Changes by Category

In [9]:
if updated_species:
    df_updated = pd.DataFrame(updated_species)
    
    # Count status transitions
    print("\n=== Status Change Patterns ===")
    status_changes = df_updated.groupby(['old_status', 'new_status']).size().reset_index(name='count')
    status_changes = status_changes.sort_values('count', ascending=False)
    display(status_changes)
    
    # Analyze by kingdom
    print("\n=== Updates by Kingdom ===")
    kingdom_counts = df_updated['kingdom'].value_counts()
    display(kingdom_counts)
    
    # Define conservation status severity order (worst to best)
    status_order = ['EX', 'EW', 'CR', 'EN', 'VU', 'NT', 'LC', 'DD', 'NE']
    
    def status_severity(status):
        if status in status_order:
            return status_order.index(status)
        return 999
    
    # Identify improvements vs deteriorations
    df_updated['old_severity'] = df_updated['old_status'].apply(status_severity)
    df_updated['new_severity'] = df_updated['new_status'].apply(status_severity)
    df_updated['change_type'] = df_updated.apply(
        lambda row: 'Improved' if row['new_severity'] > row['old_severity'] 
        else ('Deteriorated' if row['new_severity'] < row['old_severity'] else 'Changed'),
        axis=1
    )
    
    print("\n=== Conservation Status Trends ===")
    change_summary = df_updated['change_type'].value_counts()
    display(change_summary)
    
    # Show deteriorated species
    deteriorated = df_updated[df_updated['change_type'] == 'Deteriorated'][[
        'scientific_name', 'common_name_en', 'old_status', 'new_status', 'kingdom', 'family'
    ]]
    if len(deteriorated) > 0:
        print(f"\n=== Species with Deteriorated Status ({len(deteriorated)}) ===")
        display(deteriorated)
    
    # Show improved species
    improved = df_updated[df_updated['change_type'] == 'Improved'][[
        'scientific_name', 'common_name_en', 'old_status', 'new_status', 'kingdom', 'family'
    ]]
    if len(improved) > 0:
        print(f"\n=== Species with Improved Status ({len(improved)}) ===")
        display(improved)


=== Status Change Patterns ===


,old_status,new_status,count
3,EN,LC,2
5,LC,NT,2
6,LR/lc,LC,2
7,LR/nt,NT,2
9,VU,NT,2
0,CR,EN,1
1,DD,EN,1
2,DD,LC,1
4,EN,VU,1
8,NT,LC,1



=== Updates by Kingdom ===


kingdom
ANIMALIA    10
PLANTAE      5
Name: count, dtype: int64


=== Conservation Status Trends ===


change_type
Deteriorated    8
Improved        7
Name: count, dtype: int64


=== Species with Deteriorated Status (8) ===


,scientific_name,common_name_en,old_status,new_status,kingdom,family
1,Belomys pearsonii,Hairy-footed Flying Squirrel,DD,LC,ANIMALIA,SCIURIDAE
2,Castanopsis kawakamii,Ca Oi Qua To,LR/nt,NT,PLANTAE,FAGACEAE
5,Disepalum petelotii,Nhoc trai khop la mac,LR/lc,LC,PLANTAE,ANNONACEAE
6,Drepananthus filiformis,,LR/lc,LC,PLANTAE,ANNONACEAE
9,Limosa lapponica,Bar-tailed Godwit,LC,NT,ANIMALIA,SCOLOPACIDAE
10,Lophura diardi,Siamese Fireback,LC,NT,ANIMALIA,PHASIANIDAE
12,Sindora tonkinensis,Go Mat,DD,EN,PLANTAE,FABACEAE
13,Teinopalpus imperialis,Kaiser-i-Hind,LR/nt,NT,ANIMALIA,PAPILIONIDAE



=== Species with Improved Status (7) ===


,scientific_name,common_name_en,old_status,new_status,kingdom,family
0,Altigena tonkinensis,Cá Hoả,VU,NT,ANIMALIA,CYPRINIDAE
3,Chelonia mydas,Green Turtle,EN,LC,ANIMALIA,CHELONIIDAE
4,Diospyros mun,Ebony,CR,EN,PLANTAE,EBENACEAE
7,Ephippiorhynchus asiaticus,Black-necked Stork,NT,LC,ANIMALIA,CICONIIDAE
8,Ketupa zeylonensis,Brown Fish-owl,EN,LC,ANIMALIA,STRIGIDAE
11,Platalea minor,Black-faced Spoonbill,EN,VU,ANIMALIA,THRESKIORNITHIDAE
14,Vanellus vanellus,Northern Lapwing,VU,NT,ANIMALIA,CHARADRIIDAE


## Export Results to CSV

In [ ]:
# Export updated species to CSV
if updated_species:
    df_updated_export = pd.DataFrame(updated_species)
    output_file = 'iucn_status_updates.csv'
    df_updated_export.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Updated species exported to {output_file}")

# Export new species to CSV
if new_species:
    df_new_export = pd.DataFrame(new_species)
    output_file = 'iucn_new_species.csv'
    df_new_export.to_csv(output_file, index=False, encoding='utf-8')
    print(f"New species exported to {output_file}")

# Export removed species to CSV
if removed_species:
    df_removed_export = pd.DataFrame(removed_species)
    output_file = 'iucn_removed_species.csv'
    df_removed_export.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Removed species exported to {output_file}")